# Day 03 Tutorial — Joins, Aggregations, Windows

**Goal:** Join types, aggregations, ranking patterns.


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
customers = spark.createDataFrame(
    [('c1', 'Alice'), ('c2', 'Bob'), ('c3', 'Carol'), ('c4', 'Dan')],
    ['customer_id', 'name'],
)
orders = spark.createDataFrame(
    [
        ('o1', 'c1', 100.0, '2024-01-01'),
        ('o2', 'c1', 200.0, '2024-01-03'),
        ('o3', 'c2', 50.0, '2024-01-02'),
        ('o4', 'c5', 80.0, '2024-01-04'),
    ],
    ['order_id', 'customer_id', 'amount', 'order_date'],
)


## Joins


In [ ]:
customers.join(orders, 'customer_id', 'inner').show()
customers.join(orders, 'customer_id', 'left').show()
customers.join(orders, 'customer_id', 'left_anti').show()
customers.join(orders, 'customer_id', 'left_semi').show()


## Aggregations and broadcast


In [ ]:
from pyspark.sql.functions import broadcast
orders.groupBy('customer_id').agg(
    F.count('*').alias('order_cnt'),
    F.sum('amount').alias('total_amount'),
).show()
broadcast(customers).join(orders, 'customer_id').show()


## Windows


In [ ]:
w = Window.partitionBy('customer_id').orderBy(F.col('order_date').desc())
orders.withColumn('rn', F.row_number().over(w)).show()
orders.withColumn('rn', F.row_number().over(w)).filter(F.col('rn') == 1).show()


## Interview
- Broadcast small dimensions.
- row_number vs rank vs dense_rank.
